# Phase 2 — Feature Engineering

Construct customer-level panel with churn flags, CLV, and RFM features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.bcs.data import load_raw, drop_missing_customers, drop_cancellations, drop_negative_quantity, drop_negative_price
from src.bcs.features import build_customer_panel, log1p_scale

In [ ]:
df = load_raw("../data/online_retail_II.xlsx")
print(f"Raw shape: {df.shape}")

In [ ]:
df, missing_rate = drop_missing_customers(df)
print(f"Dropped {missing_rate:.1%} rows with missing Customer ID")
print(f"After drop: {df.shape}")

In [ ]:
cancellation_rate = df["Invoice"].astype(str).str.startswith("C").mean()
print(f"Cancellation rate: {cancellation_rate:.1%}")
print(f"Cancelled invoices: {df['Invoice'].astype(str).str.startswith('C').sum()}")

In [ ]:
df = drop_cancellations(df)
df = drop_negative_quantity(df)
df = drop_negative_price(df)
print(f"After cleaning: {df.shape}")

In [ ]:
panel, n_segments = build_customer_panel(
    df,
    churn_window_days=90,
    segment_col="Country",
    min_segment_size=15,
)
print(f"Total customers: {len(panel)}")
print(f"Segments: {n_segments}")
print(panel.head())

In [ ]:
seg_summary = panel.groupby("segment_name").agg(
    n_customers=("Customer ID", "count"),
    churn_rate=("churned", "mean"),
    mean_clv=("clv", "mean"),
    median_clv=("clv", "median"),
).sort_values("n_customers", ascending=False)
print(seg_summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
churn_by_seg = panel.groupby("segment_name")["churned"].mean().sort_values(ascending=False)
churn_by_seg.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by country segment")
ax.axhline(panel["churned"].mean(), color="red", linestyle="--", label=f"Overall ({panel['churned'].mean():.1%})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(panel["clv"].describe())
print(f"Skewness: {panel['clv'].skew():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panel["clv"].hist(bins=50, ax=axes[0])
axes[0].set_title("CLV distribution (raw)")
axes[0].set_xlabel("Total revenue")

np.log1p(panel["clv"]).hist(bins=50, ax=axes[1])
axes[1].set_title("CLV distribution (log1p)")
axes[1].set_xlabel("log1p(total revenue)")
plt.tight_layout()
plt.show()

In [ ]:
panel.to_parquet("../data/customer_panel.parquet", index=False)
print("Panel saved to data/customer_panel.parquet")